# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking
Document: D&D 5e SRD spell chapter (`data/spells.md`).
Bi-encoder `all-MiniLM-L6-v2` + ChromaDB (cosine), cross-encoder `ms-marco-MiniLM-L-6-v2`, LLM `gemini-flash-latest`.
Both pipelines share the exact same chunks and the same top-10 candidate list.

In [19]:
import os
import re
import json
import time
import gc
import glob
import shutil
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
import subprocess
from google import genai
from google.genai import types
from sentence_transformers import CrossEncoder
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter


DATA_DIR = "data"
CHROMA_DIR = "./chroma_db_full"      
RESULTS_FILE = "results.json"
BI_ENCODER = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL = "gemini-3.7-flash"
TOP_K = 10    
TOP_N = 3    
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Loading Data into Database
Only `data/spells.md` is indexed: 500-character chunks with 50 overlap, embedded with `all-MiniLM-L6-v2` and stored in ChromaDB (cosine). The store is deleted and rebuilt on every run so chunks are never duplicated.

In [ ]:
file_path = "data/spells.md"
with open(file_path, "r", encoding="utf-8-sig") as f:
    markdown_content = f.read()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.create_documents([markdown_content])
embedding_model = HuggingFaceEmbeddings(
    model_name=BI_ENCODER,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory=CHROMA_DIR,
    collection_metadata={"hnsw:space": "cosine"},   # cosine distance
)
print(f"Successfully added {len(docs)} chunks from {file_path} into ChromaDB! Store now holds {vector_store._collection.count()} vectors.")

In [3]:
load_dotenv()
cross_encoder = CrossEncoder(CROSS_ENCODER, device=device)   

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5966.37it/s]


# Shared helpers (retrieval, prompt, LLM call)

In [9]:
def sync():
    if device == "cuda":
        torch.cuda.synchronize()


def get_top_10_chunks(query):
    raw_results = vector_store.similarity_search_with_score(query, k=10)
    
    formatted_docs = []
    for rank, (doc, score) in enumerate(raw_results, start=1):
       
        doc.metadata["bi_rank"] = rank
        doc.metadata["ce_rank"] = rank 
        doc.metadata["cosine"] = float(score) 
        doc.metadata["ce_score"] = float(score) 
        doc.metadata["section"] = doc.metadata.get("section", "Spells Overview")
        doc.metadata["source"] = "spells.md"
        
        formatted_docs.append(doc)
        
    return formatted_docs


def build_prompt(query: str, ctx_docs):
    """One prompt template for both pipelines."""
    context_text = "\n\n---\n\n".join([d.page_content for d in ctx_docs])
    return f"""You are a helpful Dungeons & Dragons 5e rules assistant. Answer the user's question accurately using only the provided context below.

Context:
{context_text}

Question: {query}
Answer:"""


# def generate_answer(prompt: str, retries: int = 6):
#     """Calls Claude CLI using -p (print mode), retrying on failures. Returns (text, seconds)."""
#     for attempt in range(retries):
#         try:
#             t0 = time.perf_counter()
            
#             # Execute claude in non-interactive print mode
#             result = subprocess.run(
#                 ["claude", "-p", prompt],
#                 capture_output=True,
#                 text=True,
#                 check=True
#             )
            
#             return result.stdout.strip(), time.perf_counter() - t0
            
#         except Exception as e:
#             wait = 10 * (attempt + 1)
#             print(f"  Claude CLI error ({type(e).__name__}), retrying in {wait}s")
#             time.sleep(wait)
            
#     return "ERROR: Claude CLI call failed", None


# Initialize your client once outside the function (or inside if needed)
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

def generate_answer(prompt: str, retries: int = 6):
    """Calls Gemini 1.5 Flash, retrying on rate limits or errors. Returns (text, seconds)."""
    for attempt in range(retries):
        try:
            t0 = time.perf_counter()
            
            response = client.models.generate_content(
                model="gemini-3.7-flash",
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0),
            )
            
            return (response.text or "").strip(), time.perf_counter() - t0
            
        except Exception as e:
            wait = 10 * (attempt + 1)
            print(f"  Gemini error ({type(e).__name__}), retrying in {wait}s: {e}")
            time.sleep(wait)
            
    return "ERROR: Gemini call failed", None

# Baseline Retrieval

In [5]:
def run_baseline_pipeline(query: str, top_10_docs):
    """
    Pipeline 1: takes the pre-retrieved Top 10 chunks, picks the Top 3 directly,
    and queries Gemini to generate Answer 1.
    """
    baseline_top_3 = top_10_docs[:TOP_N]
    answer, t_llm = generate_answer(build_prompt(query, baseline_top_3))
    return baseline_top_3, answer, {"llm_s": t_llm}

# Advance RE-ranking

In [6]:
def rerank_chunks(query: str, top_10_docs):
    """Scores every (query, chunk) pair with the cross-encoder and re-sorts descending."""
    t0 = time.perf_counter()
    scores = cross_encoder.predict([[query, d.page_content] for d in top_10_docs], show_progress_bar=False)
    sync()
    elapsed = time.perf_counter() - t0
    for d, s in zip(top_10_docs, scores):
        d.metadata["ce_score"] = float(s)
    reranked = sorted(top_10_docs, key=lambda d: d.metadata["ce_score"], reverse=True)
    for rank, d in enumerate(reranked, 1):
        d.metadata["ce_rank"] = rank
    return reranked, elapsed


def run_advanced_pipeline(query: str, top_10_docs):
    """
    Pipeline 2: takes the same pre-retrieved Top 10 chunks, re-scores them with the Cross-Encoder,
    picks the new Top 3, and queries Gemini (identical prompt) to generate Answer 2.
    """
    reranked, t_rerank = rerank_chunks(query, top_10_docs)
    reranked_top_3 = reranked[:TOP_N]
    answer, t_llm = generate_answer(build_prompt(query, reranked_top_3))
    return reranked_top_3, answer, {"rerank_s": t_rerank, "llm_s": t_llm}

In [7]:
def run_both_pipelines(query: str):
    """
    Retrieves the Top 10 chunks once, then feeds the same candidates
    to the baseline and the re-ranking pipelines.
    """
    t0 = time.perf_counter()
    top_10_docs = get_top_10_chunks(query)
    t_retrieve = time.perf_counter() - t0
    baseline_top_3, answer_1, t1 = run_baseline_pipeline(query, top_10_docs)
    reranked_top_3, answer_2, t2 = run_advanced_pipeline(query, top_10_docs)
    return {
        "top_10": top_10_docs,
        "baseline_top_3": baseline_top_3, "answer_1": answer_1,
        "reranked_top_3": reranked_top_3, "answer_2": answer_2,
        "latency_s": {"retrieve": t_retrieve, "rerank": t2["rerank_s"],
                      "llm_p1": t1["llm_s"], "llm_p2": t2["llm_s"]},
    }

# for _ in range(3):
#     rerank_chunks("warm up", get_top_10_chunks("warm up"))
# demo = run_both_pipelines("What is the casting time, range, and damage type of the Fireball spell?")
# print("ANSWER 1:", demo["answer_1"], "\n\nANSWER 2:", demo["answer_2"])

# Run both pipelines on the questions in questions.json
`questions.json` has 20 spell questions with a short gold answer each (no gold passages). Two checks are stored per question:
- **Chunk relevance (proxy):** a retrieved chunk counts as relevant if it contains the gold answer text. This lets us track the rank of the first answer-bearing chunk before and after re-ranking.
- **Answer correctness (rough):** the gold answer (or >= 75% of its content words) appears in the LLM answer, for Answer 1 and Answer 2.

Everything is written to `results.json` (resumable: finished questions are skipped on re-run).

In [8]:
NUM = {"zero": "0", "one": "1", "two": "2", "three": "3", "four": "4", "five": "5",
       "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10"}
STOP = {"a", "an", "the", "of", "to", "or", "and", "your", "you", "is", "are"}


def norm_tokens(text):
    return [NUM.get(w, w) for w in re.findall(r"[a-z0-9]+", text.lower())]


def contains_answer(text, gold):
    """Proxy for chunk relevance: the gold answer (or one of its 'or' alternatives) appears in the text."""
    t = " " + " ".join(norm_tokens(text)) + " "
    alts = [" ".join(norm_tokens(a)) for a in re.split(r"\s+or\s+", gold, flags=re.I)]
    return any(a and f" {a} " in t for a in alts)


def answer_correct(answer, gold):
    """Rough correctness: gold answer appears in the LLM answer, or >= 75% of its content words do."""
    if contains_answer(answer, gold):
        return True
    words = [w for w in norm_tokens(gold) if w not in STOP]
    got = set(norm_tokens(answer))
    return bool(words) and sum(w in got for w in words) / len(words) >= 0.75


def first_rank(docs, key, flags):
    """Best (lowest) value of `key` among relevant chunks, or None."""
    ranks = [d.metadata[key] for d, f in zip(docs, flags) if f]
    return min(ranks) if ranks else None


def record_for(query, flags_fn):
    """Runs both pipelines on one query and builds the JSON record (flags_fn(doc) -> relevant?)."""
    out = run_both_pipelines(query)
    top10 = out["top_10"]
    flags = [flags_fn(d) for d in top10] if flags_fn else [None] * len(top10)
    rec = {
        "top10": [{
            "bi_rank": d.metadata["bi_rank"], "ce_rank": d.metadata["ce_rank"],
            "cosine": round(d.metadata["cosine"], 4), "ce_score": round(d.metadata["ce_score"], 3),
            "section": d.metadata["section"], "source": d.metadata["source"],
            "relevant": f, "text": d.page_content,
        } for d, f in zip(top10, flags)],
        "pipeline_1_baseline": {"context_bi_ranks": [d.metadata["bi_rank"] for d in out["baseline_top_3"]],
                                "answer": out["answer_1"]},
        "pipeline_2_reranked": {"context_bi_ranks": [d.metadata["bi_rank"] for d in out["reranked_top_3"]],
                                "answer": out["answer_2"]},
        "latency_ms": {k: (round(v * 1000, 1) if v is not None else None) for k, v in out["latency_s"].items()},
    }
    if flags_fn and any(f is not None for f in flags):
        b = first_rank(top10, "bi_rank", flags)
        r = first_rank(top10, "ce_rank", flags)
        rec["rank_shift"] = {
            "first_relevant_baseline_rank": b, "first_relevant_reranked_rank": r,
            "relevant_in_top10": b is not None,
            "baseline_has_relevant_in_top3": b is not None and b <= TOP_N,
            "reranked_has_relevant_in_top3": r is not None and r <= TOP_N,
            "promoted_into_top3": b is not None and b > TOP_N and r <= TOP_N,
            "demoted_out_of_top3": b is not None and b <= TOP_N and r > TOP_N,
        }
    return rec


with open("questions.json", "r", encoding="utf-8") as f:
    questions = [{"qid": f"q{item['id']}", "question": item["question"], "gold_answer": item["answer"]}
                 for item in json.load(f)]
print(len(questions), "questions")

20 questions


In [ ]:

store = {"config": {}, "qa_results": [], "failure_queries": []}
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r", encoding="utf-8") as f:
        store = json.load(f)
store["config"] = {"bi_encoder": BI_ENCODER, "cross_encoder": CROSS_ENCODER, "llm": LLM_MODEL,
                   "vector_store": "ChromaDB (cosine)", "num_chunks": len(docs), "chunk_size": 500,
                   "chunk_overlap": 50, "top_k": TOP_K, "top_n": TOP_N, "device": device,
                   "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None}


def save_store():
    with open(RESULTS_FILE, "w", encoding="utf-8") as f:
        json.dump(store, f, indent=2, ensure_ascii=False)


def ok(rec):
    return not rec["pipeline_1_baseline"]["answer"].startswith("ERROR") and \
           not rec["pipeline_2_reranked"]["answer"].startswith("ERROR")


done = {r["qid"] for r in store["qa_results"] if ok(r)}
store["qa_results"] = [r for r in store["qa_results"] if r["qid"] in done]

for i, q in enumerate(questions, 1):
    if q["qid"] in done:
        continue
    print(f"[{i}/{len(questions)}] {q['qid']}: {q['question'][:90]}")
    flags_fn = lambda d, g=q["gold_answer"]: contains_answer(d.page_content, g)
    rec = record_for(q["question"], flags_fn)
    rec = {"qid": q["qid"], "question": q["question"], "gold_answer": q["gold_answer"],
           "pipeline_1_correct": answer_correct(rec["pipeline_1_baseline"]["answer"], q["gold_answer"]),
           "pipeline_2_correct": answer_correct(rec["pipeline_2_reranked"]["answer"], q["gold_answer"]), **rec}
    store["qa_results"].append(rec)
    save_store()
    time.sleep(2)   
save_store()
print("saved", len(store["qa_results"]), "records to", RESULTS_FILE)

# Results of the runs


In [25]:
RESULTS_FILE = "results.json"  
with open(RESULTS_FILE, "r", encoding="utf-8") as f:
    results_data = json.load(f)

qa_results = results_data.get("qa_results", [])

total_questions = len(qa_results)
if total_questions > 0:
    baseline_correct = sum(1 for r in qa_results if r.get("pipeline_1_correct"))
    reranked_correct = sum(1 for r in qa_results if r.get("pipeline_2_correct"))
    
    baseline_acc = (baseline_correct / total_questions) * 100
    reranked_acc = (reranked_correct / total_questions) * 100
    
    print(f"=== Pipeline Evaluation Summary ({total_questions} Questions) ")
    print(f"Pipeline 1 (Baseline Bi-Encoder Accuracy): {baseline_acc:.1f}% ({baseline_correct}/{total_questions})")
    print(f"Pipeline 2 (Cross-Encoder Reranked Accuracy): {reranked_acc:.1f}% ({reranked_correct}/{total_questions})")
else:
    print("No evaluation records found in results.json yet.")

summary_list = []
for r in qa_results:
    summary_list.append({
        "Question": r.get("question"),
        "Baseline Correct": r.get("pipeline_1_correct"),
        "Reranked Correct": r.get("pipeline_2_correct")

    })

summary_df = pd.DataFrame(summary_list)
display(summary_df)

=== Pipeline Evaluation Summary (19 Questions) 
Pipeline 1 (Baseline Bi-Encoder Accuracy): 63.2% (12/19)
Pipeline 2 (Cross-Encoder Reranked Accuracy): 84.2% (16/19)


,Question,Baseline Correct,Reranked Correct
0,What spell level are cantrips considered to be?,True,True
1,Which school of magic has typical effects that prevent or reverse harmful ef...,False,True
2,What are the physical requirements a spellcaster must meet to cast a spell a...,True,True
3,What action do most spells require to cast?,True,True
4,How much longer does the Ritual version of a spell take to cast compared to ...,False,True
5,What action must you take to try to identify an ongoing non-instantaneous sp...,True,True
6,What check and DC are required to identify an ongoing spell?,True,True
7,What is the formula for calculating the Spell save DC?,True,True
8,What is the formula for calculating the Spell attack modifier?,True,True
9,How many level 1 spell slots does a level 3 Wizard have?,True,True


Based on the evaluation results in your `results.json` file, here is a structured Failure Mode Analysis addressing each specific point with concrete examples from your D&D 5e RAG pipeline:

---

### 1. Keyword Trap / Distractor

* **Definition:** A scenario where a query shares high lexical overlap with an irrelevant passage, tricking the Bi-Encoder vector search into prioritizing it, while the true answer uses distinct terminology or structure.
* **Example from Data (Query Q18):**
* **Query:** *"How much Fire damage does the Fire Bolt cantrip deal on a hit?"*

* **Baseline Failure:** The Bi-Encoder was distracted by numerous other spell descriptions containing overlapping terms like "damage" and spell-slot scaling notes, burying the true *Fire Bolt* chunk down at **rank 10**. The baseline LLM failed, reporting that the damage was not mentioned.


* **Cross-Encoder Resolution:** The Cross-Encoder analyzed the deep semantic token alignment and successfully **promoted the true chunk from rank 10 to rank 1** into the Top 3 window, allowing Pipeline 2 to accurately output `1d10 Fire damage`.





---

### 2. Exceptions & Corner Cases

* **Definition:** The baseline successfully retrieves a general rule chunk, but misses a specific restriction, penalty, or conditional exception ("except when...", higher-level slot modifiers, or special interactions) located elsewhere in the text.
* **Example from Data (Query Q5):**
* **Query:** *"How much longer does the Ritual version of a spell take to cast compared to normal?"*

* **Gold Answer:** `10 minutes longer`

* **Baseline Failure:** The Bi-Encoder retrieved general sections discussing longer casting times and minutes/hours at ranks 1–4, while missing the specific rule chunk defining the 10-minute ritual rule, pushing it down to **rank 5**. The baseline LLM could only state vaguely that it requires "minutes or even hours".


* **Cross-Encoder Resolution:** The Cross-Encoder evaluated the precise semantic phrasing of the ritual rule and **promoted the chunk from rank 5 to rank 1**, enabling Pipeline 2 to correctly answer `10 minutes longer`.





---

### 3. Negation & Constraints

* **Definition:** Queries involving strict constraints, exclusions, or rules where Bi-Encoder vector embeddings blur positive and negative contexts together (e.g., matching a general rule when a query asks about prohibitions or limitations).
* **Example from Data (Query Q17):**
* **Query:** *"What is the base AC of a creature under the Mage Armor spell?"*

* **Gold Answer:** `13 plus its Dexterity modifier`

* **Constraint Challenge:** The Bi-Encoder can easily get confused by other armor or AC rules if negative constraints (such as *"who isn't wearing armor"*) are blurred. For instance, if a character is wearing armor, the spell fails early.
* **Cross-Encoder Resolution:** The Cross-Encoder strictly filters out distractor rules about heavy/medium armor training and locks onto the explicit conditional constraint chunk (*"You touch a willing creature who isn't wearing armor... base AC becomes 13 plus its Dexterity modifier"*), keeping it securely at **rank 1**.





---

### 4. Other Document-Specific Failures (Buried Treasure / Rank Promotion)

* **Definition:** Any scenario where a vital, correct chunk *was* fetched by the initial vector search, but was buried deep down at ranks 4–10, making it invisible to the LLM until successfully promoted into the Top 3 by the Cross-Encoder.
* **Example from Data (Query Q14):**
* **Query:** *"What is the range of the Fireball spell?"*

* **Gold Answer:** `150 feet`

* **Baseline Failure:** The Bi-Encoder ranked generic fire-sphere descriptions ahead of the specific *Fireball* block, burying the correct entry at **rank 9**. The baseline LLM failed, stating the range was not mentioned.


* **Cross-Encoder Resolution:** The Cross-Encoder identified the exact match for *Fireball* and **promoted the chunk from rank 9 to rank 1**, allowing Pipeline 2 to successfully surface the answer (`150 feet`).

# Part 3 tables: rank shifts, Answer 1 vs Answer 2

In [27]:

rows = []
for r in store["failure_queries"]:
    rs = r["rank_shift"]
    rows.append({"category": r["category"], "query": r["query"], "target chunk": r["target_section"],
                 "baseline rank": rs["first_relevant_baseline_rank"],
                 "re-ranked rank": rs["first_relevant_reranked_rank"],
                 "in LLM top-3 (P1/P2)": f"{rs['baseline_has_relevant_in_top3']} / {rs['reranked_has_relevant_in_top3']}"})
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 220)
pd.DataFrame(rows)

""


In [ ]:

for r in store["failure_queries"]:
    print("=" * 100)
    print(f"[{r['category']}] {r['query']}")
    print("P1 context (bi-encoder ranks):", r["pipeline_1_baseline"]["context_bi_ranks"])
    print("ANSWER 1:", r["pipeline_1_baseline"]["answer"])
    print("P2 context (bi-encoder ranks):", r["pipeline_2_reranked"]["context_bi_ranks"])
    print("ANSWER 2:", r["pipeline_2_reranked"]["answer"], "\n")

In [ ]:
#
qa = [r for r in store["qa_results"] if "rank_shift" in r]
df = pd.DataFrame([{"qid": r["qid"], **r["rank_shift"]} for r in qa])
n = len(df)
print(f"{n} questions")
print(f"LLM answer correct (rough check)  Pipeline 1: {sum(r['pipeline_1_correct'] for r in qa)}/{n}   Pipeline 2: {sum(r['pipeline_2_correct'] for r in qa)}/{n}")
print(f"relevant chunk somewhere in top 10 (ceiling for re-ranking): {df['relevant_in_top10'].mean():.1%}")
print(f"Hit@3  baseline: {df['baseline_has_relevant_in_top3'].mean():.1%}   re-ranked: {df['reranked_has_relevant_in_top3'].mean():.1%}")
mrr = lambda col: df[col].apply(lambda x: 1 / x if x else 0).mean()
print(f"MRR@10 baseline: {mrr('first_relevant_baseline_rank'):.3f}   re-ranked: {mrr('first_relevant_reranked_rank'):.3f}")
print(f"promoted into top 3: {df['promoted_into_top3'].sum()}   demoted out of top 3: {df['demoted_out_of_top3'].sum()}")
df[df["promoted_into_top3"] | df["demoted_out_of_top3"]]

19 questions
LLM answer correct (rough check)  Pipeline 1: 12/19   Pipeline 2: 16/19
relevant chunk somewhere in top 10 (ceiling for re-ranking): 84.2%
Hit@3  baseline: 57.9%   re-ranked: 78.9%
MRR@10 baseline: 0.733   re-ranked: 0.944
promoted into top 3: 4   demoted out of top 3: 0


,qid,first_relevant_baseline_rank,first_relevant_reranked_rank,relevant_in_top10,baseline_has_relevant_in_top3,reranked_has_relevant_in_top3,promoted_into_top3,demoted_out_of_top3
1,q2,5.0,1.0,True,False,True,True,False
4,q5,5.0,1.0,True,False,True,True,False
13,q14,9.0,1.0,True,False,True,True,False
17,q18,10.0,1.0,True,False,True,True,False


# Latency profiling

In [30]:
lat = pd.DataFrame([r["latency_ms"] for r in store["qa_results"] + store["failure_queries"]])
lat["pipeline_1_retrieval"] = lat["retrieve"]
lat["pipeline_2_retrieval"] = lat["retrieve"] + lat["rerank"]
lat["pipeline_1_end_to_end"] = lat["retrieve"] + lat["llm_p1"]
lat["pipeline_2_end_to_end"] = lat["retrieve"] + lat["rerank"] + lat["llm_p2"]
table = lat.agg(["mean", "median", "max"]).T.round(1)
print(f"{len(lat)} queries, device: {device}")
table

19 queries, device: cuda


,mean,median,max
retrieve,18.5,10.2,125.8
rerank,22.4,23.4,28.6
llm_p1,3920.4,3444.3,9743.4
llm_p2,3489.2,2914.8,13033.9
pipeline_1_retrieval,18.5,10.2,125.8
pipeline_2_retrieval,40.9,33.5,150.2
pipeline_1_end_to_end,3938.9,3462.4,9869.2
pipeline_2_end_to_end,3530.1,2947.0,13066.7
